In [2]:
import numpy as np 
from sentence_transformers import SentenceTransformer

embed_model=SentenceTransformer("all-MiniLM-L6-v2")

def cosine_similarity(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))


words=["automobile", "vehicle", "dinosaur", "stick"]
embeddings={word: embed_model.encode(word) for word in words}

print("Similarity (automobile vs vehicle):",
      cosine_similarity(embeddings["automobile"], embeddings["vehicle"]))
print("Similarity (automobile vs dinosaur):", 
      cosine_similarity(embeddings["automobile"],embeddings["dinosaur"]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity (automobile vs vehicle): 0.84713644
Similarity (automobile vs dinosaur): 0.39764318


In [3]:
import json 
import pandas as pd

with open("embedding_index_3m.json", "r") as f:
    data=json.load(f)
    df=pd.DataFrame(data)

    print("generating local embedding for transcripts....")
    embed_model=SentenceTransformer("all-MiniLM-L6-v2")
    df["local_vector"]=list(
        embed_model.encode(df["summary"].tolist(), show_progress_bar=True)

    )

    def cosine_similarity(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    def get_videos(query,top_k=3):
        query_vector=embed_model.encode(query)

        df["similarity"]=df["local_vector"].apply(
            lambda x: cosine_similarity(query_vector,x)
        )

        results=df.sort_values("similarity",ascending=False).head(top_k)
        return results 

generating local embedding for transcripts....


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

In [4]:
import os
from groq import Groq

groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Fetch all available models for your API key
models = groq_client.models.list()
print("--- YOUR ACTIVE GROQ MODELS ---")
for model in models.data:
    print(f"- {model.id}")

--- YOUR ACTIVE GROQ MODELS ---
- openai/gpt-oss-20b
- whisper-large-v3
- canopylabs/orpheus-arabic-saudi
- openai/gpt-oss-safeguard-20b
- qwen/qwen3.8-27b
- openai/gpt-oss-120b
- meta-llama/llama-prompt-guard-2-86m
- allam-2-7b
- canopylabs/orpheus-v1-english
- meta-llama/llama-prompt-guard-2-22m
- whisper-large-v3-turbo


In [5]:
import os
from groq import Groq

# Initialize Groq client
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


def search_and_summarize(query):
    # 1. Retrieve top matching video segments
    results = get_videos(query, top_k=2)

    context = ""
    print("\n--- MATCHING YOUTUBE CLIPS FOUND ---")
    for idx, row in results.iterrows():
        url = f"https://www.youtube.com/watch?v={row['videoId']}&t={row['start']}s"
        print(f"Title: {row['title']}")
        print(f"Link: {url}")
        print(f"Similarity Score: {row['similarity']:.4f}\n")
        context += (
            f"Video Title: {row['title']}\nSummary: {row['summary']}\nURL:"
            f" {url}\n\n"
        )

    # 2. Synthesize answer using an active Groq model
    prompt = f"""
    You are an AI assistant answering questions based on YouTube video transcripts.
    
    User Query: "{query}"
    
    Retrieved Context:
    {context}
    
    Please provide a concise answer to the user's question and cite the relevant video links.
    """

    # Updated working model string
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",  # <--- UPDATED MODEL STRING
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )

    print("--- AI RESPONSE (Groq) ---")
    print(response.choices[0].message.content)


# Run search and generate summary
search_and_summarize("What is Azure Machine Learning ?")


--- MATCHING YOUTUBE CLIPS FOUND ---
Title: Data Science, Convolutional Neural Networks, and Machine Learning in the Cloud (Part 3 of 4)
Link: https://www.youtube.com/watch?v=0TwbqkQ9pxk&t=00:06:05s
Similarity Score: 0.8574

Title: What’s new with Azure Machine Learning
Link: https://www.youtube.com/watch?v=YlWCeY_CWEg&t=00:15:14s
Similarity Score: 0.8288

--- AI RESPONSE (Groq) ---
Azure Machine Learning is Microsoft’s cloud‑based platform that lets data scientists and developers build, train, register, and deploy machine‑learning models at scale. It provides end‑to‑end pipelines, model‑registry capabilities, MLOps‑ready release pipelines, and native integration with tools such as Power BI for data enrichment and visualization.  

- **Video 1** – “Data Science, Convolutional Neural Networks, and Machine Learning in the Cloud (Part 3 of 4)” discusses how Azure ML lets users register models, manage them, and deploy them in the cloud.  
  <https://www.youtube.com/watch?v=0TwbqkQ9pxk&t=0

In [6]:
test_queries = [
    "How do convolutional neural networks work?",
    "What is ONNX runtime and how is it used?",
    "Explain automated machine learning in Azure",
]

for q in test_queries:
    print(f"\n==================== QUERY: {q} ====================")
    search_and_summarize(q)


==================== QUERY: How do convolutional neural networks work? ====================

--- MATCHING YOUTUBE CLIPS FOUND ---
Title: Demystifying AI
Link: https://www.youtube.com/watch?v=k-K3g4FKS_c&t=00:03:03s
Similarity Score: 0.6442

Title: An Intuitive Approach to Machine Learning Models (Part 1 of 4)
Link: https://www.youtube.com/watch?v=lPyK38sRWLI&t=00:09:09s
Similarity Score: 0.6097

--- AI RESPONSE (Groq) ---
Convolutional neural networks (CNNs) learn to recognize patterns in images by stacking a series of layers that progressively extract higher‑level features:

1. **Convolutional layers** slide small filters (kernels) over the input image, computing dot products that highlight local patterns such as edges, colors, or textures.  
2. **Non‑linear activation functions** (e.g., ReLU) introduce non‑linearity so the network can model complex relationships.  
3. **Pooling layers** down‑sample the feature maps, reducing dimensionality and providing translation invariance.  
4. 

Add Threshold Filtering (Precision Guardrail)

In [7]:
def get_video_with_threshold(query, pd_vector, top_k, threshold=0.40):
    query_vector=embed_model.encode(query)

    pd_vector["Similarity"]=pd_vector["local_vector"].apply(
        lambda x:cosine_similarity(query_vector,x)
    )

    filtered_df=pd_vector[pd_vector["similarity"]>=threshold]

    return filtered_df.sort_values("similarity",ascending=False).head

In [8]:
test_query = "What is Azure Machine Learning?"

result=get_video_with_threshold(test_query,df,top_k=3,threshold=0.40)
print(result)

<bound method NDFrame.head of                                             speaker  \
1267               Cesar De la Torre Llorente, Seth   
1243                               Seth Juarez, PSI   
748                                           Chris   
23                                      Seth Juarez   
277                                    Parashar Sha   
...                                             ...   
1294                                  Baiju, Cassie   
382   Corom Thomposon, Joseph Suraj, Karim Thompson   
1269               Cesar De la Torre Llorente, Seth   
1236        Rob Nunez, Albert Dankwa, Ashish Bhatia   
1086                               Adina Trufinescu   

                                                  title      videoId  \
1267  Build Recap | What’s new in Azure Machine Lear...  tXrDscVaF4Q   
1243     Simplifying Machine Learning with Automated ML  samWIFxMHj4   
748              What’s new with Azure Machine Learning  YlWCeY_CWEg   
23    Data Science, C

In [9]:
# Test 2: Run with an irrelevant query to test if the threshold guardrail blocks it
irrelevant_query = "How to bake a chocolate chip cookie"
results_filtered = get_video_with_threshold(irrelevant_query, df, top_k=3, threshold=0.40)

if callable(results_filtered):
    results_filtered = results_filtered()
if hasattr(results_filtered, "head"):
    results_filtered = results_filtered.head(3)

print(f"Irrelevant query results found: {len(results_filtered) if isinstance(results_filtered, pd.DataFrame) else 0}")
# This should print 0 rows (or significantly fewer rows) because cookie baking won't match AI transcripts closely!

Irrelevant query results found: 3


Build a Conversational Loop with Chat Memory

In [ ]:
from groq.types.chat import ChatCompletionMessageParam


chat_history: list[ChatCompletionMessageParam] = []


def chat_with_transcript_search():
    print("YouTube Transcript Search Engine (Groq Powered)")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        user_query = input("Ask a question: ").strip()
        if user_query.lower() in ["exit", "quit"]:
            print("Session ended.")
            break

        if not user_query:
            continue

        # 1. Retrieve matching contexts
        results = get_videos(user_query, top_k=2)

        context = ""
        for idx, row in results.iterrows():
        # 2. Build system context and message payload
        url = f"https://www.youtube.com/watch?v={row['videoId']}&t={row['start']}s"
        context += (
            f"Video Title: {row['title']}\n"
            f"Summary: {row['summary']}\n"
            f"URL: {url}\n\n"
        )

        # Build message history for the request
        messages: list[ChatCompletionMessageParam] = [
        {
            "role": "system",
            "content": (
            "You are a helpful YouTube assistant. Use the retrieved context "
            "to answer the user's question with video citations."
            ),
        }
        ]

        # Add past turn history
        messages.extend(chat_history)

        # Add current turn with retrieved context
        current_prompt = f"Context:\n{context}\n\nUser Question: {user_query}"
        messages.append({"role": "user", "content": current_prompt})
        context += (
            f"Video Title: {row['title']}\n"
            f"Summary: {row['summary']}\n"
            f"URL: {url}\n\n"
        )

    # 2. Build system context and message payload
    messages: list[ChatCompletionMessageParam] = [
        {
            "role": "system",
            "content": (
                "You are a helpful YouTube assistant. Use the retrieved context "
                "to answer the user's question with video citations."
            ),
        }
    ]

        # Add past turn history
        ]

        # Add past turn history
        messages.extend(chat_history)

        # Add current turn with retrieved context
        current_prompt = f"Context:\n{context}\n\nUser Question: {user_query}"
        messages.append({"role": "user", "content": current_prompt})

        # 3. Request completion from Groq[cite: 11]
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",  # Active model string
            messages=messages,
            temperature=0.2,
        )

        answer = response.choices[0].message.content
        print(f"\nAI: {answer}\n" + "-" * 50)

        # Update chat memory
        chat_history.append({"role": "user", "content": user_query})
        chat_history.append({"role": "assistant", "content": answer})


# Run interactive session
chat_with_transcript_search()